# Eigenbasis ViT — depth vs. width at a fixed parameter budget

Sweep `2026-10-02_19-43-18G3352432P4193954`, 6x6 $J_1$ ($J_2 = 0$, `sign_rule = FF`,
cluster eigenbasis, `embedding = qn`, no point-group projection). Four ViT shapes
zipped over `model.layers` / `model.ratio` / `model.heads`, chosen so the
parameter count stays roughly constant while depth and width trade off:

| run | layers | ratio | heads | $d_\mathrm{model}$ | n_params |
|---|---|---|---|---|---|
| 0 | 1 | 6 | 15 | 90 | 99 585 |
| 1 | 2 | 6 | 11 | 66 | 98 406 |
| 2 | 4 | 6 | 8 | 48 | 99 840 |
| 3 | 8 | 8 | 4 | 32 | 87 520 |

So this answers one question: **at ~100k parameters, is it better to be wide and
shallow or narrow and deep?** The 8-layer column is 12% smaller than the rest,
which is worth keeping in mind before reading too much into its gap.

**Two caveats that shape how far these numbers go.**

1. `iter = 2000` — short. The 2026-09-23 8-layer run on this same system needed
   20 000 steps to reach a relative error of $9\times10^{-4}$. What is measured
   here is *how fast each shape descends in its first 2000 steps*, not where it
   ends up.
2. `auto_alpha = false` with `alpha = 1.25`, i.e. overdispersed importance
   sampling held fixed well away from the Born value of 2. The weights are
   therefore non-trivial and `ESS_w` is a real cost here (it runs at 9-16%),
   unlike a plain-Born run where it is pinned at 100%.

All four runs completed their target in a single window — no `run step NNNN.log`
resubmission files, so `run.log` is the whole history.

In [ ]:
from pathlib import Path
import json, re
import numpy as np
import matplotlib.pyplot as plt

# "data/eigenbasis" sits in the repo root above the folder holding this notebook, so the same discovery
# works locally and on the cluster (as in plot_energy_vs_iteration.ipynb).
def _find_data_root(name="data/eigenbasis"):
    here = Path.cwd().resolve()
    for base in (here, *here.parents):
        if (base / name).is_dir():
            return base / name
    raise FileNotFoundError(f"could not find a {name!r} directory at or above {here}")

DATA_ROOT = _find_data_root()

# Pinned on purpose: this notebook is about one sweep, so a later upload should
# not silently change what it plots. Swap the name to point it elsewhere.
SWEEP = "2026-10-02_19-43-18G3352432P4193954"
SWEEP_DIR = DATA_ROOT / SWEEP

# 6x6, J2 = 0 reference (per site). ED is skipped at this size, so E0 is null in
# run.summary.json and this literal is the only reference available.
E0_PER_SITE = -0.678872

SMOOTH = 51         # rolling-mean window in steps; the runs are only 2000 long

print("sweep:", SWEEP_DIR.name)
print("exists:", SWEEP_DIR.is_dir())

## Load

In [ ]:
def _column(block, name):
    """NetKet writes a complex series as {"real": [...], "imag": [...]} and a
    real one as a plain list."""
    col = block[name]
    if isinstance(col, dict):
        col = col["real"]
    return np.asarray(col, dtype=float)


def _merge_logs(run_dir):
    """Every *.log in a run dir -> one curve sorted by global iteration.

    These runs finished in one window so there is only run.log, but resumed
    sweeps write 'run step NNNN.log' alongside it with global iteration numbers,
    and merging costs nothing.
    """
    series = {}
    for lp in sorted(run_dir.glob("*.log")):
        E = json.load(open(lp))["Energy"]
        for i, m, s, v, rh in zip(E["iters"], _column(E, "Mean"),
                                  _column(E, "Sigma"), _column(E, "Variance"),
                                  _column(E, "R_hat")):
            series[int(i)] = (m, s, v, rh)
    it = np.array(sorted(series))
    cols = np.array([series[i] for i in it], dtype=float)
    return it, cols[:, 0], cols[:, 1], cols[:, 2], cols[:, 3]


def _diag(run_dir):
    p = run_dir / "run.diag.jsonl"
    if not p.exists():
        return {}
    rows = [json.loads(l) for l in p.open() if l.strip()]
    keys = ("step", "var_eloc_per_site", "ess_w_frac", "rhat_logpsi", "acceptance")
    return {k: np.array([r.get(k, np.nan) for r in rows], dtype=float) for k in keys}


def _from_stdout(run_dir):
    """Parameter count, d_model and wall-clock minutes, as the driver printed
    them. run.summary.json carries neither n_params nor the runtime."""
    txt = (run_dir / "stdout.txt").read_text(errors="replace")
    def grab(pat, cast, default=None):
        m = re.search(pat, txt)
        return cast(m.group(1)) if m else default
    return {
        "n_params": grab(r"n_params = (\d+)", int),
        "d_model":  grab(r"d_model=(\d+)", int),
        "minutes":  grab(r"Runtime:\s*([0-9.]+)", float),
    }


def _status(run_dir):
    for flag, name in (("run.done", "completed"),
                       ("run.timeout", "timeout"),
                       ("run.frozen", "frozen")):
        if (run_dir / flag).exists():
            return name
    return "no sentinel"


matrix = json.load(open(SWEEP_DIR / "matrix.json"))
runs = []
for spec in matrix["runs"]:
    run_dir = SWEEP_DIR / f"run_{spec['run_id']:04d}"
    it, E, sigma, var, rhat = _merge_logs(run_dir)
    cfg = spec["config"]
    m = cfg["model"]
    info = _from_stdout(run_dir)
    runs.append({
        "id": spec["run_id"], "dir": run_dir,
        "layers": m["layers"], "ratio": m["ratio"], "heads": m["heads"],
        "d_model": info["d_model"] or m["ratio"] * m["heads"],
        "n_params": info["n_params"], "minutes": info["minutes"],
        "n_sites": cfg["system"]["L"] ** 2, "target": m["iter"],
        "status": _status(run_dir),
        "iters": it, "E": E, "sigma": sigma, "var": var, "rhat": rhat,
        "diag": _diag(run_dir),
    })
runs.sort(key=lambda r: r["layers"])

# Categorical slots 1-4 of the reference palette, in canonical order, assigned
# by depth so the colour means the same thing in every figure. Four curves, each
# also direct-labelled, so identity is never carried by colour alone.
PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]
for slot, r in enumerate(runs):
    r["color"] = PALETTE[slot]

TAIL = 200          # the runs are 2000 steps, so this is the last 10%

def _tail(a, cap=TAIL):
    return a[-max(1, min(cap, len(a) // 4)):]

for r in runs:
    tail = _tail(r["E"]) / r["n_sites"]
    r["e_final"], r["e_std"] = tail.mean(), tail.std()
    r["rel_err"] = abs(r["e_final"] - E0_PER_SITE) / abs(E0_PER_SITE)
    r["var_final"] = _tail(r["var"]).mean() / r["n_sites"]
    d = r["diag"]
    for key, name in (("ess_w_frac", "ess_final"), ("acceptance", "acc_final")):
        r[name] = float(np.nanmean(_tail(d[key], 10))) if d else np.nan
    # Only the total runtime is recorded, so minutes-per-step is a run average.
    # Fine for comparing shapes; it does not resolve warm-up vs steady state.
    r["s_per_it"] = r["minutes"] * 60 / max(1, len(r["iters"]))
    r["minutes_axis"] = r["iters"] * r["s_per_it"] / 60

def label(r):
    return (f"{r['layers']} layer{'s' if r['layers'] > 1 else ''}, "
            f"d={r['d_model']}  [{r['n_params']:,}p]")

for r in runs:
    print(f"run_{r['id']:04d}  layers={r['layers']:>2} ratio={r['ratio']} "
          f"heads={r['heads']:>2} d_model={r['d_model']:>3}  "
          f"{r['n_params']:>7,}p  {len(r['iters'])}/{r['target']} steps  "
          f"{r['status']}  {r['minutes']:.0f} min")

## Results

`E/site` is the mean over the final 200 logged steps — one VMC step is a noisy
estimator, so the last point alone is not a run's energy.

In [ ]:
hdr = (f"{'run':>4} {'layers':>7} {'d_model':>8} {'n_params':>9} {'E/site':>11} "
       f"{'+-std':>9} {'rel.err':>9} {'Var/site':>10} {'ESS_w':>7} {'acc':>6} "
       f"{'min':>6} {'s/it':>6}")
print(hdr)
print("-" * len(hdr))
for r in runs:
    print(f"{r['id']:>4} {r['layers']:>7} {r['d_model']:>8} {r['n_params']:>9,} "
          f"{r['e_final']:>11.6f} {r['e_std']:>9.1e} {r['rel_err']:>9.2e} "
          f"{r['var_final']:>10.2e} {r['ess_final']:>7.3f} {r['acc_final']:>6.2f} "
          f"{r['minutes']:>6.0f} {r['s_per_it']:>6.2f}")

best = min(runs, key=lambda r: r["rel_err"])
cheap = min(runs, key=lambda r: r["rel_err"] * r["minutes"])
print(f"\nreference E0/site = {E0_PER_SITE:.6f}  (6x6, J2 = 0)")
print(f"best accuracy at 2000 steps : {best['layers']} layers, d={best['d_model']}"
      f"  (rel.err {best['rel_err']:.2e})")
print(f"best accuracy per minute    : {cheap['layers']} layers, d={cheap['d_model']}"
      f"  (rel.err {cheap['rel_err']:.2e} in {cheap['minutes']:.0f} min)")

## Energy per site vs. iteration

Raw trace at low opacity, centred rolling mean on top, log iteration axis. The
right panel is the same data zoomed onto the three deeper runs; the 1-layer run
never enters that window.

In [ ]:
def smooth(y, w):
    w = min(w, max(1, len(y) // 4) * 2 + 1)
    if w < 3:
        return y
    k = np.ones(w) / w
    pad = w // 2
    return np.convolve(np.pad(y, pad, mode="edge"), k, mode="valid")


fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.8),
                         gridspec_kw={"width_ratios": [1.15, 1]})
for ax in axes:
    for r in runs:
        y = r["E"] / r["n_sites"]
        ax.plot(r["iters"], y, color=r["color"], lw=0.6, alpha=0.18)
        ax.plot(r["iters"], smooth(y, SMOOTH), color=r["color"], lw=1.8,
                label=label(r), solid_capstyle="round")
    ax.axhline(E0_PER_SITE, color="0.35", ls=":", lw=1.2, zorder=0)
    ax.grid(alpha=0.2, lw=0.6)
    ax.set_axisbelow(True)
    ax.set_xlabel("iteration")
    for sp in ("top", "right"):
        ax.spines[sp].set_visible(False)

axes[0].set_xscale("log")
axes[0].set_xlim(1, 2.4e3)
axes[0].set_ylim(-0.70, -0.40)
axes[0].set_ylabel(r"energy per site  $\langle H\rangle / N$")
axes[0].set_title("full history (log iteration)", fontsize=10)
axes[0].legend(fontsize=8.5, loc="upper right", frameon=False)
axes[0].annotate(f"$E_0/N$ = {E0_PER_SITE:.6f}", xy=(0.02, E0_PER_SITE),
                 xycoords=("axes fraction", "data"), ha="left", va="bottom",
                 fontsize=8, color="0.35")

axes[1].set_xlim(0, 2.05e3)
axes[1].set_ylim(-0.6800, -0.6690)
axes[1].xaxis.set_major_locator(plt.MaxNLocator(5))
axes[1].set_title("zoom on the deeper runs", fontsize=10)
# rank within the curves that actually land in the window, or the out-of-window
# 1-layer run eats an offset slot and the rest overprint
ends = sorted(((r["iters"][-1], smooth(r["E"] / r["n_sites"], SMOOTH)[-1],
                r["color"], f"{r['layers']}L") for r in runs
               if -0.6800 < smooth(r["E"] / r["n_sites"], SMOOTH)[-1] < -0.6690),
              key=lambda t: t[1])
for rank, (x, y, c, txt) in enumerate(ends):
    dy = (rank - (len(ends) - 1) / 2) * 11
    axes[1].annotate(f" {txt}", xy=(x, y), xytext=(4, dy),
                     textcoords="offset points", ha="left", va="center",
                     fontsize=8.5, color=c)
axes[1].annotate(f"$E_0/N$ = {E0_PER_SITE:.6f}", xy=(0.5, E0_PER_SITE),
                 xycoords=("axes fraction", "data"), ha="center", va="top",
                 fontsize=8, color="0.35")

fig.suptitle("Eigenbasis ViT depth/width scan — 6x6, $J_2=0$, 2000 steps",
             x=0.01, ha="left", fontsize=12, y=1.02)
fig.tight_layout()
plt.show()

## Relative error vs. iteration

$|E - E_0| / |E_0|$ against the per-site reference. Smoothed only — the raw
quantity spikes on a log axis wherever the trace crosses the reference.

In [ ]:
fig, ax = plt.subplots(figsize=(8.8, 4.8))
ends = []
for r in runs:
    rel = smooth(np.abs(r["E"] / r["n_sites"] - E0_PER_SITE) / abs(E0_PER_SITE),
                 SMOOTH)
    ax.loglog(r["iters"], rel, color=r["color"], lw=1.9, label=label(r),
              solid_capstyle="round")
    ends.append((r["iters"][-1], rel[-1], r["color"]))
for rank, (x, y, c) in enumerate(sorted(ends, key=lambda t: t[1])):
    dy = (rank - (len(ends) - 1) / 2) * 11
    ax.annotate(f" {y:.1e}", xy=(x, y), xytext=(6, dy),
                textcoords="offset points", ha="left", va="center",
                fontsize=8.5, color=c)
ax.set(xlabel="iteration", ylabel=r"$|E - E_0| \,/\, |E_0|$",
       title="Relative error vs. iteration")
ax.margins(x=0.16, y=0.10)      # room for the staggered endpoint labels
ax.grid(alpha=0.2, lw=0.6, which="both")
ax.set_axisbelow(True)
for sp in ("top", "right"):
    ax.spines[sp].set_visible(False)
ax.legend(fontsize=8.5, loc="lower left", frameon=False)
fig.tight_layout()
plt.show()

## The trade-off

Left: accuracy against depth, which is the sweep's own axis — $d_\mathrm{model}$
shrinks as layers grow, so moving right spends the same parameter budget on depth
instead of width.

Right: the same runs against **wall-clock minutes** rather than iterations. Cost
per step rises monotonically with depth here (3.4 → 5.0 s/it), so equal-iteration
and equal-compute comparisons are not the same ranking. Minutes are
iteration $\times$ (total runtime / steps), i.e. a run-average rate — the logs
carry no per-step timing, so this cannot resolve warm-up from steady state.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.8))

ax = axes[0]
xs = [r["layers"] for r in runs]
ys = [r["rel_err"] for r in runs]
ax.plot(xs, ys, color="0.6", lw=1.2, zorder=1)
for r in runs:
    ax.scatter(r["layers"], r["rel_err"], s=110, color=r["color"], zorder=3)
    ax.annotate(f"  d={r['d_model']}\n  {r['rel_err']:.1e}",
                xy=(r["layers"], r["rel_err"]), xytext=(10, 0),
                textcoords="offset points", ha="left", va="center",
                fontsize=8.5, color=r["color"])
ax.set_xscale("log", base=2)
ax.set_yscale("log")
ax.set_xticks(xs)
ax.set_xticklabels([str(x) for x in xs])
ax.set_xlim(0.8, 16)
ax.set(xlabel="layers  (width shrinks to the right)",
       ylabel=r"$|E - E_0| \,/\, |E_0|$  (final 200 steps)",
       title="Accuracy vs. depth at ~100k parameters")

ax = axes[1]
for r in runs:
    rel = smooth(np.abs(r["E"] / r["n_sites"] - E0_PER_SITE) / abs(E0_PER_SITE),
                 SMOOTH)
    ax.semilogy(r["minutes_axis"], rel, color=r["color"], lw=1.9,
                label=label(r), solid_capstyle="round")
    ax.annotate(f" {r['s_per_it']:.1f} s/it",
                xy=(r["minutes_axis"][-1], rel[-1]), xytext=(5, 0),
                textcoords="offset points", ha="left", va="center",
                fontsize=8, color=r["color"])
ax.set(xlabel="wall-clock minutes", ylabel=r"$|E - E_0| \,/\, |E_0|$",
       title="Accuracy vs. compute")
ax.set_xlim(0, max(r["minutes"] for r in runs) * 1.18)
ax.legend(fontsize=8.5, loc="upper right", frameon=False)

for ax in axes:
    ax.grid(alpha=0.2, lw=0.6, which="both")
    ax.set_axisbelow(True)
    for sp in ("top", "right"):
        ax.spines[sp].set_visible(False)
fig.tight_layout()
plt.show()

## Sampling health

From `run.diag.jsonl`, one panel per quantity (shared x, never a second y-axis).
`ESS_w` is included here, unlike a Born-sampled run: `alpha` is pinned at 1.25,
so the importance weights are non-trivial and the effective sample fraction is
the price being paid for the overdispersed proposal.

In [ ]:
DIAG_PANELS = [
    ("var_eloc_per_site",  r"Var$(E_{loc})/N$",  "log",  None),
    ("ess_w_frac",         r"ESS$_w$ fraction",  None,   (0, 0.6)),
    ("acceptance",         "acceptance",         None,   (0, 0.45)),
    ("rhat_logpsi",        r"$\hat{R}$  $\log|\psi|$",  None,  (0.995, 1.05)),
]
DIAG_SMOOTH = 5     # diag rows land every 20 steps -> ~100 steps

fig, axes = plt.subplots(len(DIAG_PANELS), 1, figsize=(9.5, 9.0), sharex=True)
for ax, (key, ylab, scale, ylim) in zip(axes, DIAG_PANELS):
    for r in runs:
        d = r["diag"]
        if not d:
            continue
        ax.plot(d["step"], d[key], color=r["color"], lw=0.7, alpha=0.20)
        ax.plot(d["step"], smooth(d[key], DIAG_SMOOTH), color=r["color"], lw=1.6,
                label=label(r), solid_capstyle="round")
    if scale:
        ax.set_yscale(scale)
    if ylim:
        ax.set_ylim(*ylim)
    ax.set_ylabel(ylab)
    ax.grid(alpha=0.2, lw=0.6)
    ax.set_axisbelow(True)
    for sp in ("top", "right"):
        ax.spines[sp].set_visible(False)

axes[3].axhline(1.0, color="0.35", ls=":", lw=1.2, zorder=0)
axes[0].legend(fontsize=8.5, loc="upper right", ncol=2, frameon=False)
axes[-1].set_xlabel("iteration")
fig.suptitle("Sampling diagnostics", x=0.01, ha="left", fontsize=12, y=0.995)
fig.tight_layout()
plt.show()

---

# 2026-10-05 — branch continuations

Four runs uploaded on 2026-10-05, all from `branch_run.py`: each resumes a stored
`run.mpack` and keeps going, with the parent's log history copied into the branch
directory so a merged curve spans the whole trajectory from step 0. Two parents,
two branches each.

| branch | parent | ckpt | from → to | override |
|---|---|---|---|---|
| 4L, lr → 0.005 | `2026-10-02…P4193954`/run_0002 | `671efb07` | 2000 → 7000 | `optimizer.initial/final_learning_rate = 0.005` |
| 4L, auto_alpha | `2026-10-02…P4193954`/run_0002 | `671efb07` | 2000 → 7000 | `distribution.auto_alpha = true` |
| 8L, +2000 | `2026-09-23…P2417599`/run_0003 | `9c159b5b` | 20000 → 22000 | none |
| 8L, +6000 | `2026-09-23…P2417599`/run_0003 | `9c159b5b` | 20000 → 26000 | `run.MAX_RUNTIME_MIN = 1380` |

The first pair continues **run 2 of the sweep above** — the 4-layer, $d=48$ shape —
so it speaks directly to caveat 1 of this notebook: `iter = 2000` measured how fast
each shape descends, not where it lands. The second pair continues the 8-layer,
$d=72$, 433k-parameter run that the same caveat cites as the long reference, and
asks what another 2000 or 6000 steps are worth once a run is already at
$9\times10^{-4}$.

All four wrote `run.done`. Three of them span two submission windows, so
`stdout.txt` carries more than one `Runtime:` line — see the loader.

**Three things to settle before reading any number below.**

1. **There is no null continuation.** Neither 4-layer branch is the parent config
   simply run on; both carry an override. "The branch improved" is therefore not
   evidence that the override helped — the parent was still descending on its own
   at the branch point. Every claim below is instead measured against the
   *parent's own pre-branch slope*, fitted on its trace just before the checkpoint.
2. **The schedules are offset-aware, so neither branch gets a warm-up spike.**
   `run_2D_eigenbasis.py` builds `lr = lambda c: _lr(c + step_offset)`, and with
   `lr_transition_steps = 400` both parents are long past the transition. The
   auto_alpha branch runs at a flat 0.02, the lr branch at a flat 0.005. The
   adaptive-α branch also starts its controller from scratch (`no alpha in
   .progress; adaptive alpha restarts from 1.25`), because its parent ran fixed-α.
3. **`smooth()` from the figures above is not used in this section.** It pads with
   `np.pad(..., mode="edge")`, which repeats the single final sample $w/2$ times
   and lets one unlucky step drag the drawn endpoint — on the 8L +6000 curve it
   pulls the endpoint to $6.2\times10^{-4}$ against a true last-200 mean of
   $7.6\times10^{-4}$, a 19% error that flatters the run. This section uses
   `rmean()`, a rolling mean with shrinking edge windows. The same bias makes the
   endpoint labels in the relative-error figure above read 3–10% low; it changes
   none of that figure's ranking.

## Load the branches

In [ ]:
import matplotlib.patheffects as pe

# ---------------------------------------------------------------- 2026-10-05
# Four branch continuations, two per parent. Everything below is read from
# parent.json / matrix.json, so the provenance in the table is the tool's own
# record and not a transcription.
#
# Colour is re-keyed in this section: the figures above paint by depth, these
# paint by *branch identity* (slots 1-4 of the same categorical palette, fixed
# order, never cycled). Every curve is also direct-labelled.
BRANCH_DIRS = [
    "2026-10-05_14-40-41Gbbe667aP1579011_branch",   # 4L, lr -> 0.005
    "2026-10-05_14-40-49Gbbe667aP1581531_branch",   # 4L, auto_alpha on
    "2026-10-05_14-18-54Gbbe667aP752248_branch",    # 8L, +2000 steps
    "2026-10-05_14-21-35Gbbe667aP845544_branch",    # 8L, +6000 steps
]

def _runtime_minutes(run_dir):
    """SUM of every 'Runtime:' line, not the first.

    stdout.txt is appended across resubmissions, so three of these four runs
    carry two windows. _from_stdout() above uses re.search and would report
    only the first window -- correct for the 10-02 sweep (single window each),
    wrong here.
    """
    txt = (run_dir / "stdout.txt").read_text(errors="replace")
    return sum(float(m) for m in re.findall(r"Runtime:\s*([0-9.]+)", txt))


def _diag_alpha(run_dir):
    """The base _diag() drops `alpha`; the adaptive-alpha story needs it."""
    p = run_dir / "run.diag.jsonl"
    rows = [json.loads(l) for l in p.open() if l.strip()]
    return np.array([r.get("alpha", np.nan) for r in rows], dtype=float)


def _block(it, y, lo, hi):
    """Mean of y over global steps [lo, hi]. One VMC step is a noisy estimator,
    so every number quoted here is a block mean, never a single point."""
    m = (it >= lo) & (it <= hi)
    return float(np.mean(y[m])), int(m.sum())


branches = []
for name in BRANCH_DIRS:
    bdir = DATA_ROOT / name
    rd = bdir / "run_0000"
    par = json.load(open(rd / "parent.json"))
    cfg = json.load(open(bdir / "matrix.json"))["runs"][0]["config"]
    it, E, sigma, var, rhat = _merge_logs(rd)
    d = _diag(rd)
    d["alpha"] = _diag_alpha(rd)
    m = cfg["model"]
    start = int(par["branch"]["start_step"])
    end = int(it[-1]) + 1
    ov = par["branch"]["overrides"]
    branches.append({
        "dir": name, "run_dir": rd,
        "parent_sweep": par["parent"]["sweep_id"],
        "parent_run": par["parent"]["run_id"],
        "ckpt": par["parent"]["checkpoint"]["sha256"][:8],
        "layers": m["layers"], "d_model": m["ratio"] * m["heads"],
        "n_sites": cfg["system"]["L"] ** 2,
        "start": start, "end": end, "new_steps": end - start,
        "overrides": ov,
        "lr": (cfg["optimizer"]["initial_learning_rate"],
               cfg["optimizer"]["final_learning_rate"]),
        "auto_alpha": bool(cfg["distribution"]["auto_alpha"]),
        "minutes": _runtime_minutes(rd),
        "done": (rd / "run.done").exists(),
        "iters": it, "E": E, "sigma": sigma, "var": var, "rhat": rhat, "diag": d,
    })

# Short tags, fixed colour order, fixed plotting order.
TAGS = ["4L  lr 0.02->0.005", "4L  auto_alpha on", "8L  +2000 steps", "8L  +6000 steps"]
for b, tag, c in zip(branches, TAGS, PALETTE):
    b["tag"], b["color"] = tag, c
    b["n_params"] = int(re.search(r"n_params = (\d+)",
                        (b["run_dir"] / "stdout.txt").read_text(errors="replace")).group(1))
    N = b["n_sites"]
    b["e_start"], _ = _block(b["iters"], b["E"] / N, b["start"] - 200, b["start"] - 1)
    b["e_end"],   _ = _block(b["iters"], b["E"] / N, b["end"] - 200, b["end"] - 1)
    b["rel_start"] = abs(b["e_start"] - E0_PER_SITE) / abs(E0_PER_SITE)
    b["rel_end"]   = abs(b["e_end"]   - E0_PER_SITE) / abs(E0_PER_SITE)
    b["s_per_it"]  = b["minutes"] * 60 / b["new_steps"]
    # Hours of wall clock spent *inside the branch*, as a branch-average rate.
    seg = b["iters"] >= b["start"]
    b["branch_hours"] = (b["iters"][seg] - b["start"]) * b["s_per_it"] / 3600
    b["rel"] = np.abs(b["E"] / N - E0_PER_SITE) / abs(E0_PER_SITE)
    # Local power law rel_err ~ step^-p over the branch segment, fitted on 40
    # block means so single-step noise does not set the slope.
    x_, y_ = b["iters"][seg], b["rel"][seg]
    idx = np.array_split(np.arange(len(x_)), 40)
    xb = np.array([x_[i].mean() for i in idx]); yb = np.array([y_[i].mean() for i in idx])
    b["p_decay"] = -np.polyfit(np.log(xb), np.log(yb), 1)[0]
    # The same fit on the PARENT's own trace just before the branch point: the
    # control this set of runs does not otherwise have. Window = the last 3000
    # steps before the branch, or half the pre-branch history if that is
    # shorter. Without it "the branch improved" is unreadable -- the parent was
    # still descending on its own.
    w_ = min(3000, b["start"] // 2)
    xp_, yp_ = b["iters"], b["rel"]
    mp_ = (xp_ >= b["start"] - w_) & (xp_ < b["start"])
    idx = np.array_split(np.arange(int(mp_.sum())), 40)
    xb_ = np.array([xp_[mp_][i].mean() for i in idx])
    yb_ = np.array([yp_[mp_][i].mean() for i in idx])
    b["p_parent"] = -np.polyfit(np.log(xb_), np.log(yb_), 1)[0]
    b["p_window"] = (int(b["start"] - w_), b["start"])

for b in branches:
    ov = ", ".join(f"{k}={v}" for k, v in b["overrides"].items()) or "none"
    print(f"{b['tag']:<14} parent {b['parent_sweep'][:16]}/run_{b['parent_run']:04d} "
          f"ckpt {b['ckpt']}  {b['start']:>5} -> {b['end']:<5} "
          f"({b['new_steps']:>4} new)  overrides: {ov}")


# --- shared drawing helpers for this section ---
INK, MUTED, PARENT = "#0b0b0b", "#52514e", "#9a9892"
# Branch segments run 2000-6000 steps, so a wider window than the 51 used for
# the 2000-step sweep above. The 8-layer family needs more: its whole signal is
# ~1.4e-4 in relative error while the per-step scatter is ~1.4e-3, so a 101-wide
# mean still carries visible noise.
BR_SMOOTH = 101


def rmean(y, w=BR_SMOOTH):
    """Centred rolling mean with SHRINKING windows at the edges.

    Not smooth(): that one pads with np.pad(..., mode="edge"), i.e. it repeats
    the single last sample w//2 times. On a trace this noisy one unlucky final
    step drags the whole tail -- on the 8L +6000 curve it pulls the drawn
    endpoint to 6.2e-4 against a true last-200 mean of 7.6e-4, a 19% error in
    the direction that flatters the run. Shrinking the window instead makes the
    endpoint the mean of the last w//2+1 samples, which is what it should be.
    """
    y = np.asarray(y, float)
    w = min(w, max(1, len(y) // 4) * 2 + 1)
    if w < 3:
        return y
    c = np.concatenate([[0.0], np.cumsum(y)])
    n, h = len(y), w // 2
    lo = np.maximum(np.arange(n) - h, 0)
    hi = np.minimum(np.arange(n) + h + 1, n)
    return (c[hi] - c[lo]) / (hi - lo)


def end_label(ax, x, y, text, color, dy=0):
    """Chip in the series colour, text in ink. Two of these four slots sit
    under 3:1 on a white surface, and coloured small text is where that bites."""
    ax.plot([x], [y], marker="o", ms=5, color=color, mec="white", mew=1.0,
            zorder=5)
    t = ax.annotate(f" {text}", xy=(x, y), xytext=(7, dy),
                    textcoords="offset points", ha="left", va="center",
                    fontsize=8.5, color=INK, zorder=6)
    # Halo so a label that has to sit over a neighbouring trace stays readable
    # without an opaque box eating the data under it.
    t.set_path_effects([pe.withStroke(linewidth=2.5, foreground="white")])

## Results

`p_par` is the control this set of runs does not otherwise contain: the same
power-law fit applied to the parent's trace just before the branch point. Read
`ratio` — a branch that descends more slowly than its parent did is an override
that cost something.

In [ ]:
hdr = (f"{'branch':<20} {'from':>6} {'to':>6} {'E/site @to':>12} {'rel @from':>10} "
       f"{'rel @to':>9} {'gain':>6} {'p_par':>6} {'p_br':>6} {'ratio':>7} "
       f"{'GPU-h':>6} {'s/it':>6}")
print(hdr); print("-" * len(hdr))
for b in branches:
    print(f"{b['tag']:<20} {b['start']:>6} {b['end']:>6} {b['e_end']:>12.6f} "
          f"{b['rel_start']:>10.2e} {b['rel_end']:>9.2e} "
          f"{b['rel_start'] / b['rel_end']:>5.2f}x {b['p_parent']:>6.2f} "
          f"{b['p_decay']:>6.2f} {b['p_decay'] / b['p_parent']:>6.2f}x "
          f"{b['minutes'] / 60:>6.1f} {b['s_per_it']:>6.1f}")

print(f"\nreference E0/site = {E0_PER_SITE:.6f}    gain = rel@from / rel@to")
print("p_br  = local exponent in rel_err ~ step^-p over the branch segment")
print("p_par = the same fit on the PARENT trace just before the branch -- the")
print("        control these four runs do not otherwise have, since none of")
print("        them is an unmodified continuation of the 4L parent.")
print("        windows: " + ", ".join(f"{b['tag'].split()[0]} {b['p_window'][0]}-{b['p_window'][1]}"
                                      for b in (branches[0], branches[2])))
print("ratio = p_br / p_par;  < 1 means the override SLOWED a descent that was")
print("        already happening.")

# ---- restart noise: the 8L pair is an accidental A/B -----------------------
# Both 8L branches resume the SAME checkpoint (sha 9c159b5b) with the same
# config and the same seed, so their overlap 20000-21999 measures what two
# nominally identical restarts differ by -- the floor any claimed improvement
# has to clear.
a, bb = branches[2], branches[3]
ia = {int(v): i for i, v in enumerate(a["iters"])}
ib = {int(v): i for i, v in enumerate(bb["iters"])}
ov = np.array(sorted(set(ia) & set(ib)))
N8 = a["n_sites"]
ea = np.array([a["E"][ia[i]] for i in ov]) / N8
eb = np.array([bb["E"][ib[i]] for i in ov]) / N8
pre, post = ov < a["start"], ov >= a["start"]

print(f"\ncopied parent history (steps {ov[pre][0]}-{ov[pre][-1]}) bit-identical: "
      f"{np.array_equal(ea[pre], eb[pre])}")
blocks = [(lo, lo + 499) for lo in range(a["start"], a["end"], 500)]
dblk = np.array([abs(ea[(ov >= lo) & (ov <= hi)].mean()
                     - eb[(ov >= lo) & (ov <= hi)].mean()) for lo, hi in blocks])
NOISE_REL = float(dblk.mean() / abs(E0_PER_SITE))
print(f"post-branch (steps {a['start']}-{a['end'] - 1}) they diverge: per-step "
      f"|dE/N| median {np.median(np.abs(ea[post] - eb[post])):.1e}")
print(f"  500-step block means differ by {dblk.mean():.1e} +- {dblk.std():.1e} per site")
print(f"  -> restart noise floor in relative error ~ {NOISE_REL:.1e}")
gap = abs(bb["rel_end"] - a["rel_end"])
print(f"  the 22k -> 26k endpoint gap is {gap:.1e}, i.e. {gap / NOISE_REL:.0f}x that floor")

## Where each branch went

Both panels plot the same quantity, but note the decade labels before comparing
them by eye: the 8-layer family is already an order of magnitude closer in, and a
shared axis would flatten it to a line. Grey is the parent history the branches
share — it is byte-identical in both directories of each pair, which is the first
check that `branch_run.py` copied rather than re-ran anything.

In [ ]:
# Smoothed only, as in the relative-error figure above: on a log axis the raw
# per-step quantity spans a decade and becomes a wash that hides the signal.
#        title, branches, window, xlim, ylim, legend loc, per-branch label dy
FAMILIES = [
    ("4 layers, $d$=48  (99,840 p)",  branches[:2], 101, (1150, 7950),
     (3.0e-3, 6.3e-3), "lower left", (0, 0)),
    ("8 layers, $d$=72  (433,296 p)", branches[2:], 201, (17800, 27600),
     (6.9e-4, 1.00e-3), "upper right", (22, 0)),
]

fig, axes = plt.subplots(1, 2, figsize=(13.0, 4.9))
for ax, (title, fam, win, xlim, ylim, legloc, labdy) in zip(axes, FAMILIES):
    bp = fam[0]["start"]
    ref = fam[0]                      # parent history is byte-identical in both
    pre = ref["iters"] < bp
    ax.plot(ref["iters"][pre], rmean(ref["rel"][pre], win),
            color=PARENT, lw=1.8, label=f"parent (shared, to {bp})", zorder=2)
    ax.axvline(bp, color=PARENT, ls=":", lw=1.2, zorder=1)
    ax.annotate(f"branch @ {bp}", xy=(bp, 1.0), xycoords=("data", "axes fraction"),
                xytext=(4, -8), textcoords="offset points",
                ha="left", va="top", fontsize=8, color=MUTED)

    for b, dy in zip(fam, labdy):
        seg = b["iters"] >= bp
        x, ys = b["iters"][seg], rmean(b["rel"][seg], win)
        ax.plot(x, ys, color=b["color"], lw=2.0, label=b["tag"],
                solid_capstyle="round", zorder=4)
        # The chip marks the quoted last-200 block mean, not the smoother's own
        # final point -- the two differ by a fraction of the window noise and
        # the label should sit on the number it states.
        end_label(ax, x[-1], b["rel_end"], f"{b['rel_end']:.2e}", b["color"], dy=dy)

    ax.set_xlim(*xlim); ax.set_ylim(*ylim); ax.set_yscale("log")
    ax.set_xlabel("global iteration")
    ax.set_title(title, fontsize=10)
    ax.legend(fontsize=8.5, loc=legloc, frameon=False)
    ax.grid(alpha=0.2, lw=0.6, which="both")
    ax.set_axisbelow(True)
    for sp in ("top", "right"):
        ax.spines[sp].set_visible(False)

axes[0].set_ylabel(r"$|E - E_0| \,/\, |E_0|$")
# Same measurement in both panels but the y-ranges differ ~5x -- the 8-layer
# family is already an order of magnitude closer in, and a shared axis would
# flatten it to a line. Read the decade labels before comparing panels by eye.
axes[1].annotate(f"two nominally identical restarts of\nthis checkpoint differ by "
                 f"~{NOISE_REL:.0e}", xy=(0.02, 0.03), xycoords="axes fraction",
                 fontsize=8, color=MUTED, ha="left", va="bottom")
fig.suptitle("2026-10-05 branch continuations — where each run went after its checkpoint",
             x=0.01, ha="left", fontsize=12, y=1.02)
fig.tight_layout()
plt.show()

## The 4-layer pair, mechanism

Same checkpoint, same 5000 steps, same wall clock to within 2 minutes — one knob
each. The energy sits above the three sampler quantities on a shared x so the
sequence is readable: α turns, the local-energy variance falls, the energy
follows. Four panels rather than twinned axes.

In [ ]:
# The 4-layer pair is the interesting one: same checkpoint, same 5000 steps,
# one knob each. This figure asks *why* they separate, so the energy sits above
# the three sampler quantities on a shared x -- four panels, never a second
# y-axis on one.
lr_b, aa_b = branches[0], branches[1]
BP, END = lr_b["start"], lr_b["end"]


def seg(b, key="rel", w=BR_SMOOTH):
    m = b["iters"] >= BP
    return b["iters"][m], rmean(b[key][m], w)


def dseg(b, key, w=9):
    """diag rows land every 20 steps, so w=9 is ~180 steps."""
    d = b["diag"]; m = d["step"] >= BP
    return d["step"][m], rmean(d[key][m], w)


# Crossover: first step after which auto_alpha stays below lr-0.005 for good.
xl, yl = seg(lr_b)
xa, ya = seg(aa_b)
common = np.intersect1d(xl, xa)
dl = np.array([yl[np.searchsorted(xl, s)] for s in common])
da = np.array([ya[np.searchsorted(xa, s)] for s in common])
ahead = da < dl
XOVER = int(common[len(ahead) - np.argmin(ahead[::-1]) if not ahead.all() else 0])

PANELS = [
    ("energy",  r"$|E - E_0|\,/\,|E_0|$", "log",  None,          None),
    ("alpha",   r"overdispersion $\alpha$", None, (1.19, 1.30),  "diag"),
    ("var",     r"Var$(E_{loc})/N$",      None,   (2.5e-3, 9e-3), "diag"),
    ("ess",     r"ESS$_w$ fraction",      None,   (0.05, 0.13),  "diag"),
]
DKEY = {"alpha": "alpha", "var": "var_eloc_per_site", "ess": "ess_w_frac"}

fig, axes = plt.subplots(4, 1, figsize=(9.6, 10.2), sharex=True,
                         gridspec_kw={"height_ratios": [1.5, 1, 1, 1]})
for ax, (key, ylab, scale, ylim, src) in zip(axes, PANELS):
    for b in (lr_b, aa_b):
        x, y = seg(b) if src is None else dseg(b, DKEY[key])
        ax.plot(x, y, color=b["color"], lw=1.9, label=b["tag"],
                solid_capstyle="round", zorder=3)
    if scale:
        ax.set_yscale(scale)
    if ylim:
        ax.set_ylim(*ylim)
    ax.axvline(XOVER, color=MUTED, ls=":", lw=1.2, zorder=1)
    ax.set_ylabel(ylab)
    ax.grid(alpha=0.2, lw=0.6, which="both")
    ax.set_axisbelow(True)
    for sp in ("top", "right"):
        ax.spines[sp].set_visible(False)

axes[0].set_ylim(3.0e-3, 5.6e-3)
for b, dy in ((lr_b, 0), (aa_b, 0)):
    end_label(axes[0], b["end"] - 1, b["rel_end"],
              f"{b['rel_end']:.2e}   $p$={b['p_decay']:.2f}", b["color"], dy=dy)
axes[0].annotate(f"parent slope before the branch: $p$={lr_b['p_parent']:.2f}",
                 xy=(0.015, 0.06), xycoords="axes fraction", fontsize=8.5,
                 color=MUTED, ha="left", va="bottom")
axes[0].legend(fontsize=9, loc="upper right", frameon=False)
axes[0].annotate(f"auto_alpha takes the lead at ~{XOVER}", xy=(XOVER, 1.0),
                 xycoords=("data", "axes fraction"), xytext=(6, -8),
                 textcoords="offset points", ha="left", va="top",
                 fontsize=8.5, color=MUTED)
# alpha is pinned, not adaptive, on the lr branch -- say so on the flat line
# rather than letting it read as "the controller chose 1.25".
axes[1].annotate("pinned (auto_alpha = false)", xy=(END - 100, 1.25),
                 xytext=(0, -12), textcoords="offset points", ha="right",
                 va="top", fontsize=8, color=INK)
axes[-1].set_xlabel("global iteration")
axes[-1].set_xlim(BP, END + 900)
fig.suptitle("4-layer branch pair — the 4x smaller step froze the run; adaptive "
             "$\\alpha$ kept it descending",
             x=0.01, ha="left", fontsize=12, y=0.997)
fig.tight_layout()
plt.show()

for b in (lr_b, aa_b):
    d = b["diag"]; m0 = (d["step"] >= BP) & (d["step"] < BP + 400); m1 = d["step"] >= END - 400
    print(f"{b['tag']:<20} Var/N {np.nanmean(d['var_eloc_per_site'][m0]):.2e} -> "
          f"{np.nanmean(d['var_eloc_per_site'][m1]):.2e}   "
          f"ESS {np.nanmean(d['ess_w_frac'][m0]):.3f} -> {np.nanmean(d['ess_w_frac'][m1]):.3f}   "
          f"alpha {np.nanmean(d['alpha'][m0]):.3f} -> {np.nanmean(d['alpha'][m1]):.3f}")
print(f"crossover at step {XOVER}")

## Cost, and whether each override actually helped

In [ ]:
# What the compute bought. Left: the same relative error against wall clock
# *spent inside the branch*, so the four are comparable as purchases rather
# than as step counts (they cost 4.4 vs 13.0 s/it). Right: the descent
# exponent before and after each override, which is the only way to read these
# branches as experiments -- none of them has an unmodified twin.
fig, axes = plt.subplots(1, 2, figsize=(13.0, 4.9),
                         gridspec_kw={"width_ratios": [1.2, 1]})

ax = axes[0]
for b in branches:
    m = b["iters"] >= b["start"]
    win = 101 if b["layers"] == 4 else 201
    ax.plot(b["branch_hours"], rmean(b["rel"][m], win), color=b["color"],
            lw=2.0, label=b["tag"], solid_capstyle="round")
    end_label(ax, b["branch_hours"][-1], b["rel_end"], f"{b['rel_end']:.2e}",
              b["color"])
ax.set_yscale("log")
ax.set_xlim(-0.6, 24.5)
ax.set(xlabel="wall-clock hours since the branch point",
       ylabel=r"$|E - E_0| \,/\, |E_0|$")
ax.set_title("What the continuation bought", fontsize=10)
ax.legend(fontsize=8.5, loc="upper right", frameon=False)
# Hours are steps x (total runtime / steps), a branch average: only the total
# is recorded, so this cannot resolve warm-up from steady state, and three of
# the four runs span two windows (one hit MAX_RUNTIME_MIN and was resubmitted).

ax = axes[1]
ypos = np.arange(len(branches))[::-1]
for y, b in zip(ypos, branches):
    ax.plot([b["p_parent"], b["p_decay"]], [y, y], color=PARENT, lw=1.6,
            zorder=1, solid_capstyle="round")
    ax.plot([b["p_parent"]], [y], marker="o", ms=7, color="white",
            mec=PARENT, mew=1.8, zorder=3)
    ax.plot([b["p_decay"]], [y], marker="o", ms=8, color=b["color"],
            mec="white", mew=1.0, zorder=4)
    faster = b["p_decay"] >= b["p_parent"]
    ax.annotate(f"{b['p_decay'] / b['p_parent']:.2f}x",
                xy=(b["p_decay"], y), xytext=(10 if faster else -10, 0),
                textcoords="offset points", ha="left" if faster else "right",
                va="center", fontsize=9, color=INK)
ax.set_yticks(ypos)
ax.set_yticklabels([b["tag"] for b in branches], fontsize=9)
ax.set_ylim(-0.6, len(branches) - 0.4)
ax.set_xlim(0, 0.78)
ax.set_xlabel(r"descent exponent $p$   in  $|E-E_0|/|E_0| \sim \mathrm{step}^{-p}$")
ax.set_title("Before (hollow = parent) vs after (filled = branch)", fontsize=10)

for ax in axes:
    ax.grid(alpha=0.2, lw=0.6, which="both")
    ax.set_axisbelow(True)
    for sp in ("top", "right"):
        ax.spines[sp].set_visible(False)
axes[1].grid(axis="y", visible=False)

fig.suptitle("Cost of the continuations, and whether each override actually helped",
             x=0.01, ha="left", fontsize=12, y=1.02)
fig.tight_layout()
plt.show()

print("steps needed to halve the relative error at each branch's own rate:")
for b in branches:
    f = 2 ** (1 / b["p_decay"])
    print(f"  {b['tag']:<20} x{f:>6.1f}  ->  step {b['end'] * f:>9,.0f}  "
          f"= {b['end'] * (f - 1) * b['s_per_it'] / 3600:>7.0f} more GPU-hours")

## What these four runs say

**1. The 2000-step ranking in this notebook was a rate measurement, and the rate
was not the whole story.** The 4-layer shape sat at $5.1\times10^{-3}$ at step
2000. Left alone with adaptive α it reaches $3.3\times10^{-3}$ by 7000 and is
still descending at $p = 0.45$ — faster than the $p = 0.25$ it had in its last
1000 steps of the sweep. Whatever the depth/width figure above ranked, it did not
rank converged runs, and the 4-layer column in particular was still accelerating
when the sweep cut it off.

**2. Dropping the learning rate 4× at step 2000 froze the run.** This is the
clearest result of the four, and it is negative. `lr 0.02 → 0.005` takes the
descent exponent from the parent's 0.25 down to **0.10** — 0.40× the rate it
already had. For the first ~1800 steps it looks like the better branch, because a
smaller step is a quieter estimator and the curve is smooth; by step 3780 the
other branch has passed it and keeps going. At $p = 0.10$ this branch needs
~890× more steps to halve its error. It is not slowly converging, it is parked.
The low `Var(E_loc)/N` in panel 3 is not sampling health — it is the variance of a
model that has stopped moving.

**3. Adaptive α nearly doubled the parent's descent rate, and the mechanism is
legible.** The controller first pushes α *down* to ~1.21 (more overdispersed) and
the run gets worse — `Var(E_loc)/N` rises to $7\times10^{-3}$ and the energy
stalls above the lr branch for 1800 steps. Then around step 3700 it reverses,
climbs to ~1.27, variance drops to $4.3\times10^{-3}$ and `ESS_w` rises
0.078 → 0.102, and the energy falls off a step. Final exponent $p = 0.45$, i.e.
1.77× the parent's. Two cautions: this is one seed and one branch point, and the
early excursion means the controller cost ~1800 steps before it paid, so a run
shorter than ~4000 steps would have recorded `auto_alpha` as a regression.

**4. The two 4-layer branches are not a clean head-to-head.** They differ in two
knobs at once (step size *and* α policy), so the gap between the blue and orange
curves is not "what adaptive α is worth". The parent-slope comparison is what
separates them: against the same control, one override is 0.40× and the other is
1.77×. That is the comparison worth quoting.

**5. The 8-layer run is still on a clean power law at 26000 — and that is the bad
news.** Both continuations reproduce the parent's exponent (0.52 → 0.56 / 0.58),
so nothing has saturated and nothing has gone wrong. But $p \approx 0.58$ means
halving $7.6\times10^{-4}$ takes ~3.3× more steps: step ~86 000, roughly **220
more GPU-hours** on top of the ~95 already spent. The 6000-step branch bought
$9.0\to7.6\times10^{-4}$ for 21.5 hours. Accuracy at this shape is now priced in
days per factor of two, and more steps is the expensive way to buy it.

**6. The 8L pair is an accidental A/B, and it sets the noise floor.** Both resume
the same checkpoint with the same config and the same seed, yet they diverge
immediately: the restart does not restore the sampler RNG state. That is a free
measurement of run-to-run spread — 500-step block means differ by
$4.3\times10^{-6}$ per site, i.e. $\sim6\times10^{-6}$ in relative error. Useful
as a yardstick: the 22k → 26k gap is 13× it (real), while anything quoted to
better than $\sim10^{-5}$ from a single run is not.

### What this suggests running next

- A **null continuation** of 10-02 run_0002, 2000 → 7000, no overrides. Six GPU-hours
  closes the one gap that weakens every 4-layer claim here.
- `auto_alpha` on the **8-layer** run. It is already on there, so the comparison
  that is missing is the reverse: fixed α at 1.25 from 20000, to see whether the
  controller is still earning its keep that deep into a run.
- If the 4-layer shape is worth pursuing, **more steps at lr 0.02 with adaptive α**
  beats anything tried here — but note it is at $3.3\times10^{-3}$ after 7000
  steps where the 4.3× larger 8-layer model is at $7.6\times10^{-4}$, so this is a
  question about the parameter budget, not about the schedule.
- Nothing here justifies a learning-rate drop at a branch point. If one is wanted
  for a final anneal, it should come much later and be checked against the parent
  slope, not against its own smoothness.